## Groubi Assignment
##### Submitted By: Nasiya Pervez (Matrikel-Nr: 253465)
##### Course Name: Optimization Implementation in Production and Logistics 
Your task is to implement a given mixed-integer linear programming model (MILP) for the flow-shop problem and to solve the attached 10 benchmark instances using Gurobi. (Bowman’s Model)

Problem Description:

In a flow shop problem, all jobs are so similar in technological nature that they all have essentially the same order of processing on the set of machines. In other words, a flow shop contains a natural machine order such that, for every job considered, operation 1 is performed on machine 1, operation 2 on machine 2, operation 3 on machine 3, etc. The task is to find a sequence of jobs on all machines.

Solve all attached instances with your model, use time limit of 5 mins in case instance is not solved to optimality. Make sure that you import the data according to the model formulation. After formulating the model, use suitable big M value, check for any infeasibilities. Upload .ipynb/py file in the Assignment section in e-learning.

You must:

    Use Bowman's Model using variable yikt

    yikt = 1, if job i is processed by machine k in period t;0, otherwise

    Objective: C(max) -> Minimize

Expected Output: 

    Makespan (Cmax): The total time required to complete all jobs on all machines. This is usually the key objective in scheduling.

    Completion times of each job on each machine (C_ik): The finish times when each job completes on each machine.

    Job scheduling decisions (e.g., start times, assignments): Which job is scheduled on which machine at what time.
    
    Precedence/order relations between jobs if applicable (e.g., which job precedes another).

Mandatory outputs from a correctly working Bowman scheduling model should at least include:

    A non-zero makespan that reflects the schedule length.

    Completion times (C_ik) with meaningful values > 0 for jobs on machines.

    A feasible job schedule showing which jobs run on which machines and when.

    If precedence constraints exist, a clear order between jobs.

# Output 
No feasible solution found within the time limit. Hence, certain modifications made.

# Modifications Made 
MIPGap of 20% (Solution 20% away from optimal)

Time Limit increased to 900 seconds (15 minutes)

Time Horizon and Value of Big M = int(n * np.max(p_ik))

# Other Modifications Can Be (Added in Comments)
Added `slack_p[i,k]` variables to relax processing time constraints.  
Introduced binary variables `relax_prec[i,k]` to allow relaxation of precedence constraints.  
Updated the objective function to include penalties for slack variables and relaxation indicators.

# Additional Notes  
- All other constraints remain the same as before.  
- The time limit is set to 15 minutes.



In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR10_5_1_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 900  # 15 minutes = 900 seconds
model.setParam('MIPGap',0.20) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")


# Gurobi Optimization Summary
#### Instance: VFR10_5_1_Gap

## Problem Summary 
- **Jobs:** 10
- **Machines:** 5

## Solver Parameters
- **Time Limit:** 900 seconds
- **MIP Gap:** 0.2

## System Information
- **Gurobi Version:** 12.0.2
- **Build:** v12.0.2rc0 (mac64[rosetta2])
- **CPU Model:** Apple M2
- **Cores Used:** 8 threads (8 logical processors)

## Model Details
- **Rows:** 136,760
- **Columns:** 47,151
- **Nonzeros:** 39,993,620
- **Variable Types:** 
  - 51 continuous  
  - 47,100 binary/integer

## Presolve Phase
- **Presolve Time:** ~60.67s
- **Final Rows:** 124,337
- **Final Columns:** 43,040
- **Final Nonzeros:** 34,265,419

## Initial Heuristic Solution
- **Objective:** 808.0000000

## Barrier Method
- **Iterations:** 2
- **Time:** ~793.75s
- **Objective Improvement Log:**
  - Iter 0: Primal = 6474.67621, Dual = -34179.6322
  - Iter 1: Primal = 6455.52756, Dual = -725371.263
  - Iter 2: Primal = 2313.99621, Dual = -2534277.91

## Root Relaxation
- **Objective:** 66.39496
- **Iterations:** 43,842
- **Time:** 718.97s

## Branch-and-Bound
- **Nodes Explored:** 1
- **Simplex Iterations:** 84,039
- **Elapsed Time:** 900.80s
- **Solution Count:** 1

## Optimization Results
- **Best Objective:** 808.0000000
- **Best Bound:** 66.39496134042
- **Gap:** 91.78%
- **Termination Reason:** Time Limit Reached

## Final Makespan
- **Makespan:** 808.00

## Completion Times (C_ik)
| Job | Machine 0 | Machine 1 | Machine 2 | Machine 3 | Machine 4 |
|-----|-----------|-----------|-----------|-----------|-----------|
| 0   | 430.00    | 467.00    | 675.00    | 741.00    | 808.00    |
| 1   | 385.00    | 436.00    | 621.00    | 687.00    | 744.00    |
| 2   | 341.00    | 429.00    | 569.00    | 614.00    | 667.00    |
| 3   | 315.00    | 410.00    | 504.00    | 580.00    | 640.00    |
| 4   | 241.00    | 327.00    | 359.00    | 441.00    | 474.00    |
| 5   | 222.00    | 286.00    | 328.00    | 391.00    | 431.00    |
| 6   | 202.00    | 258.00    | 268.00    | 295.00    | ...       |

(Note: Some values truncated for brevity)

---
## 📋 Job Schedule (y[i,k,t]=1)
Each job executes on the machines in the following time periods:

Job 0: M0 [386–430], M1 [437–467], M2 [622–675], M3 [688–741], M4 [745–808]  
Job 1: M0 [342–385], M1 [430–436], M2 [570–621], M3 [622–687], M4 [688–744]  
Job 2: M0 [316–341], M1 [411–429], M2 [505–569], M3 [581–614], M4 [641–667]  
Job 3: M0 [242–315], M1 [328–410], M2 [411–504], M3 [505–580], M4 [581–640]  
Job 4: M0 [223–241], M1 [287–327], M2 [329–359], M3 [392–441], M4 [442–474]  
Job 5: M0 [203–222], M1 [259–286], M2 [287–328], M3 [329–391], M4 [403–431]  
Job 6: M0 [180–202], M1 [229–258], M2 [259–268], M3 [269–295], M4 [377–402]  
Job 7: M0 [134–179], M1 [210–228], M2 [244–254], M3 [255–259], M4 [341–376]  
Job 8: M0 [058–133], M1 [134–209], M2 [210–243], M3 [244–249], M4 [250–340]  
Job 9: M0 [001–057], M1 [058–088], M2 [089–121], M3 [122–129], M4 [130–148]  



---

## 🔁 Precedence Order (z[i,j]=1 means i precedes j)

Job 0 ← Jobs 1, 2, 3, 4, 5, 6, 7, 8, 9  
Job 1 ← Jobs 2, 3, 4, 5, 6, 7, 8, 9  
Job 2 ← Jobs 3, 4, 5, 6, 7, 8, 9  
Job 3 ← Jobs 4, 5, 6, 7, 8, 9  
Job 4 ← Jobs 5, 6, 7, 8, 9  
Job 5 ← Jobs 6, 7, 8, 9  
Job 6 ← Jobs 7, 8, 9  
Job 7 ← Jobs 8, 9  
Job 8 ← Job 9  


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR10_5_2_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 900  # 15 minutes = 900 seconds
model.setParam('MIPGap',0.20) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")


## Gurobi Optimization Summary
#### Instance: VFR10_5_2_Gap

**Loaded:** 10 jobs, 5 machines  
**Set Parameters:**  
- TimeLimit = 900  
- MIPGap = 0.2  

**Gurobi Optimizer version:** 12.0.2 (mac64[rosetta2] - Darwin 22.1.0 22A380)  
**CPU model:** Apple M2  
**Thread count:** 8 physical cores, 8 logical processors, using up to 8 threads

### Model Summary
- Rows: 144,010  
- Columns: 49,651  
- Nonzeros: 44,348,370  
- Variable types: 51 continuous, 49,600 integer (binary)  

### Coefficient Statistics
- Matrix range: [1e+00, 1e+03]  
- Objective range: [1e+00, 1e+00]  
- Bounds range: [1e+00, 1e+00]  
- RHS range: [1e+00, 1e+03]  

### Presolve
- Removed 13,420 rows and 4,449 columns  
- Presolve time: 72.43s  

### Presolved Model
- Rows: 130,590  
- Columns: 45,202  
- Nonzeros: 37,781,662  
- Variable types: 51 continuous, 45,151 integer (binary)  
- Found heuristic solution: objective = 798.0000000  

---

### Barrier Log (Root)
- Elapsed ordering time: 215.76s  
- Barrier statistics:
  - Dense cols: 819  
  - AA' NZ: 1.505e+08  
  - Factor NZ: 6.245e+08 (~5.0 GB memory)  
  - Factor Ops: 7.897e+12 (~130s/iteration)  
  - Threads: 6  

| Iter | Primal Obj | Dual Obj | Primal Residual | Dual Residual | Complementarity | Time |
|------|------------|----------|------------------|----------------|------------------|------|
| 0    | 7466.78613 | -34832.5056 | 8.40e+05 | 1.12e-03 | 2.03e+03 | 401s |
| 1    | 7702.27447 | -76422.9047 | 8.08e+05 | 2.01e+01 | 1.96e+03 | 687s |

- Barrier performed 1 iteration in 791.99s  
- Solved with dual simplex  

---

### Root Relaxation
- Objective: 65.59009  
- Iterations: 39,711  
- Time: 701.75s  

---

### MIP Summary
| Nodes | Explored | Depth | IntInf | Incumbent Obj | Best Bound | Gap |
|-------|----------|-------|--------|----------------|------------|-----|
| 0     | 1        | 0     | 7359   | 798.00000      | 65.59009   | 91.8% |

- Explored 1 node (72,468 simplex iterations) in 900.84s  
- Thread count: 8  
- Solution count: 1  
- Time limit reached  
- Best objective: 798.00  
- Best bound: 65.59009  
- Gap: 91.7807%  

---

## Final Result

**Makespan:** 798.00

### Completion Times (C<sub>ik</sub>)

```plaintext
Job 0: [474.00, 604.00, 614.00, 672.00, 798.00]
Job 1: [395.00, 537.00, 603.00, 624.00, 746.00]
Job 2: [355.00, 497.00, 546.00, 585.00, 692.00]
Job 3: [307.00, 404.00, 477.00, 574.00, 613.00]
Job 4: [291.00, 381.00, 458.00, 572.00, 575.00]
Job 5: [253.00, 275.00, 401.00, 499.00, 554.00]
Job 6: [177.00, 262.00, 302.00, ...]

## Job Schedule (`y[i,k,t]=1`)

| Job | Machine 0       | Machine 1       | Machine 2       | Machine 3       | Machine 4       |
|-----|------------------|------------------|------------------|------------------|------------------|
| 0   | Periods 396–474 | Periods 538–604 | Periods 605–614 | Periods 625–672 | Periods 747–798 |
| 1   | Periods 356–395 | Periods 498–537 | Periods 547–603 | Periods 604–624 | Periods 693–746 |
| 2   | Periods 308–355 | Periods 405–497 | Periods 498–546 | Periods 575–585 | Periods 614–692 |
| 3   | Periods 292–307 | Periods 382–404 | Periods 459–477 | Periods 573–574 | Periods 576–613 |
| 4   | Periods 254–291 | Periods 292–381 | Periods 402–458 | Periods 500–572 | Periods 573–575 |
| 5   | Periods 178–253 | Periods 263–275 | Periods 303–401 | Periods 402–499 | Periods 500–554 |
| 6   | Periods 105–177 | Periods 178–262 | Periods 263–302 | Periods 303–322 | Periods 323–407 |
| 7   | Periods 71–104  | Periods 105–110 | Periods 112–138 | Periods 140–192 | Periods 193–213 |
| 8   | Periods 33–70   | Periods 71–76   | Periods 77–111  | Periods 112–139 | Periods 140–183 |
| 9   | Periods 1–32    | Periods 33–43   | Periods 44–54   | Periods 55–88   | Periods 89–115  |

Precedence Order (z[i,j]=1 means i precedes j)
Job 0 ← Jobs 1, 2, 3, 4, 5, 6, 7, 8, 9
Job 1 ← Jobs 2, 3, 4, 5, 6, 7, 8, 9
Job 2 ← Jobs 3, 4, 5, 6, 7, 8, 9
Job 3 ← Jobs 4, 5, 6, 7, 8, 9
Job 4 ← Jobs 5, 6, 7, 8, 9
Job 5 ← Jobs 6, 7, 8, 9
Job 6 ← Jobs 7, 8, 9
Job 7 ← Jobs 8, 9
Job 8 ← Job 9



In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR10_5_3_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 900  # 15 minutes = 900 seconds
model.setParam('MIPGap',0.20) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")


### Gurobi Optimization Output
#### Instance: VFR10_5_3_Gap

**Loaded:** 10 jobs, 5 machines  
**Parameters Set:**  
- `TimeLimit = 900`  
- `MIPGap = 0.2`

**Gurobi Optimizer Version:** 12.0.2 (mac64 [rosetta2] - Darwin 22.1.0 22A380)  
**CPU Model:** Apple M2  
**Threads:** 8 physical cores, 8 logical processors  

---

#### Model Summary

- **Rows:** 144,010  
- **Columns:** 49,651  
- **Nonzeros:** 44,348,370  
- **Variable Types:**  
  - 51 continuous  
  - 49,600 integer (binary)  

**Coefficient Statistics:**
- Matrix range: [1e+00, 1e+03]  
- Objective range: [1e+00, 1e+00]  
- Bounds range: [1e+00, 1e+00]  
- RHS range: [1e+00, 1e+03]  

---

### Presolve

- Removed 14,839 rows and 4,914 columns  
- Presolve time: **58.69s**  
- Presolved model:  
  - Rows: 129,171  
  - Columns: 44,737  
  - Nonzeros: 37,232,655  

**Variable Types After Presolve:**  
- 51 continuous  
- 44,686 integer (binary)

---

### Optimization Log

- Heuristic solution found: **Objective = 849.00**

**Barrier Log:**
- Ordering time: **984.41s**  
- Barrier performed 0 iterations in **1101.85s**

**Root Relaxation:**
- Time limit reached  
- Iterations: 9,946  
- Time: **1028.04s**

**Search Summary:**
- Nodes explored: 1  
- Iterations: 9,946  
- Time: **1102.96s**  
- Solution count: 1  
- Best Objective: **849.00**  
- Best Bound: **0.00**  
- **Gap: 100.00%**  

---

## Final Solution

### Makespan: **849.00**

---

### Completion Times (C<sub>ik</sub>):

| Job | Machine 0 | Machine 1 | Machine 2 | Machine 3 | Machine 4 |
|-----|-----------|-----------|-----------|-----------|-----------|
| 0   | 511.00    | 584.00    | 668.00    | 711.00    | 849.00    |
| 1   | 412.00    | 425.00    | 583.00    | 674.00    | 755.00    |
| 2   | 321.00    | 415.00    | 511.00    | 610.00    | 645.00    |
| 3   | 227.00    | 318.00    | 363.00    | 493.00    | 501.00    |
| 4   | 198.00    | 315.00    | 341.00    | 419.00    | 425.00    |
| 5   | 181.00    | 283.00    | 309.00    | 386.00    | 390.00    |
| 6   | 140.00    | 251.00    | 256.00    | 336.00    | 348.00    |
| 7   | 133.00    | 197.00    | 239.00    | 322.00    | 344.00    |
| 8   | 87.00     | 136.00    | 224.00    | 308.00    | 341.00    |
| 9   | 24.00     | 70.00     | 109.00    | 147.00    | 230.00    |

---

## 📋 Job Schedule (`y[i,k,t]=1`)  
Each job executes on the machines in the following time periods:

Job 0: M0 [086–120], M1 [124–167], M2 [210–254], M3 [266–320], M4 [325–391]  
Job 1: M0 [062–085], M1 [088–123], M2 [167–209], M3 [210–265], M4 [298–324]  
Job 2: M0 [050–061], M1 [070–087], M2 [121–166], M3 [167–209], M4 [265–297]  
Job 3: M0 [032–049], M1 [050–069], M2 [087–120], M3 [130–166], M4 [200–264]  
Job 4: M0 [022–031], M1 [035–049], M2 [063–086], M3 [087–129], M4 [167–199]  
Job 5: M0 [012–021], M1 [022–034], M2 [035–062], M3 [063–086], M4 [130–166]  
Job 6: M0 [008–011], M1 [012–021], M2 [022–034], M3 [035–062], M4 [087–129]  
Job 7: M0 [005–007], M1 [008–011], M2 [012–021], M3 [022–034], M4 [063–086]  
Job 8: M0 [002–004], M1 [005–007], M2 [008–011], M3 [012–021], M4 [035–062]  
Job 9: M0 [000–001], M1 [002–004], M2 [005–007], M3 [008–011], M4 [012–034] 

## 🔁 Precedence Order (`z[i,j]=1` means *i* precedes *j*)

Job 0 ← Jobs 1, 2, 3, 4, 5, 6, 7, 8, 9  
Job 1 ← Jobs 2, 3, 4, 5, 6, 7, 8, 9  
Job 2 ← Jobs 3, 4, 5, 6, 7, 8, 9  
Job 3 ← Jobs 4, 5, 6, 7, 8, 9  
Job 4 ← Jobs 5, 6, 7, 8, 9  
Job 5 ← Jobs 6, 7, 8, 9  
Job 6 ← Jobs 7, 8, 9  
Job 7 ← Jobs 8, 9  
Job 8 ← Job 9  


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR10_5_4_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 900  # 15 minutes = 900 seconds
model.setParam('MIPGap',0.20) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")


### Gurobi Optimization Output Summary 
#### Instance: VFR10_5_4_Gap.txt - Run time: 43 minutes, 17.4 seconds before interruption

**Loaded:**  
- Jobs: 10  
- Machines: 5  
- Run time: 43 minutes, 17.4 seconds before interruption

**Parameters Set:**  
- `TimeLimit`: 900  
- `MIPGap`: 0.2  

**Gurobi Optimizer Version:**  
- 12.0.2 build v12.0.2rc0 (mac64[rosetta2] - Darwin 22.1.0 22A380)  
- CPU model: Apple M2  
- Thread count: 8 physical cores, 8 logical processors, using up to 8 threads  

---

### Model Overview

- Rows: 138,210  
- Columns: 47,651  
- Nonzeros: 40,846,570  
- Variable Types:  
  - Continuous: 51  
  - Integer (Binary): 47,600  

**Coefficient Statistics:**  
- Matrix range: [1e+00, 1e+03]  
- Objective range: [1e+00, 1e+00]  
- Bounds range: [1e+00, 1e+00]  
- RHS range: [1e+00, 1e+03]  

---

### Presolve

Performed iteratively over time:  
- Rows removed: 13,547  
- Columns removed: 4,507  
- Final presolve time: 55.17s  
- Presolved Model:  
  - Rows: 124,663  
  - Columns: 43,144  
  - Nonzeros: 34,551,083  
  - Binary Variables: 43,093  

**Heuristic Solution Found:**  
- Objective: 911.0000000  

---

### Root Barrier Log

**Concurrent LP Optimizer Used:**  
- Primal simplex  
- Dual simplex  
- Barrier  

**Elapsed Ordering Time:**  
- Final Ordering Time: 377.38s  

**Barrier Statistics:**  
- AA' NZ: 1.588e+08  
- Factor NZ: 6.263e+08 (~5.0 GB memory)  
- Factor Ops: 7.012e+12 (~180s per iteration)  
- Threads: 6  

**Barrier Iteration Log:**  

| Iter | Primal Obj | Dual Obj | Primal Residual | Dual Residual | Complementarity | Time |
|------|------------|----------|------------------|----------------|------------------|------|
| 0    | 7538.53328 | -33832.2608 | 8.18e+05      | 8.07e-04       | 2.05e+03         | 587s |

- Barrier performed 0 iterations in 735.40s (881.02 work units)  
- Solve interrupted – model solved by another algorithm  
- Concurrent spin time: 5.33s  

---

### Root Relaxation

- Objective: 60.07778  
- Iterations: 37,424  
- Time: 661.14s (933.64 work units)  

---

### Branch-and-Bound

| Nodes | Current Node | Depth | IntInf | Incumbent Obj | Best Bound | Gap  | Time |
|-------|---------------|-------|--------|----------------|------------|------|------|
| 0     | 0             | 0     | 8060   | 911.00000      | 60.07778   | 93.4%| 837s |

- Nodes explored: 1  
- Simplex iterations: 69,473  
- Time: 837.75s  
- Work units: 1416.45  

**Solution Count:** 1  
- Best Objective: 911.00000  
- Best Bound: 60.07777972784  
- Gap: 93.4053%  

---

### Final Status

**Optimization Status:** 11 (Solve Interrupted)

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR10_5_5_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 900  # 15 minutes = 900 seconds
model.setParam('MIPGap',0.20) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")


### Gurobi Optimization Output Summary
#### Instance: VFR10_5_5_Gap
**Loaded**: 10 jobs, 5 machines  
**Set parameter** `TimeLimit` to value `900`  
**Set parameter** `MIPGap` to value `0.2`  
**Gurobi Optimizer version** 12.0.2 build v12.0.2rc0 (mac64[rosetta2] - Darwin 22.1.0 22A380)

**CPU model**: Apple M2  
**Thread count**: 8 physical cores, 8 logical processors, using up to 8 threads

**Non-default parameters**:
- `TimeLimit = 900`
- `MIPGap = 0.2`

**Optimize a model with** 141110 rows, 48651 columns and 42579470 nonzeros  
**Model fingerprint**: `0x05983a3d`  
**Variable types**: 51 continuous, 48600 integer (48600 binary)

**Coefficient statistics**:
- Matrix range: [1e+00, 1e+03]
- Objective range: [1e+00, 1e+00]
- Bounds range: [1e+00, 1e+00]
- RHS range: [1e+00, 1e+03]

Presolve removed 15181 rows and 4714 columns  
**Presolve time**: 60.46s  
Presolved model: 125929 rows, 43937 columns, 35419119 nonzeros  
Variable types: 51 continuous, 43886 integer (43886 binary)  
Found heuristic solution: objective **906.0000000**

**Deterministic concurrent LP optimizer**: primal simplex, dual simplex, and barrier  
**Showing barrier log only...**

**Root barrier log...**  
Elapsed ordering time: 94.68s

**Barrier statistics**:
- AA' NZ: 1.609e+08
- Factor NZ: 6.590e+08 (approx. 5.0 GB of memory)
- Factor Ops: 7.745e+12 (approx. 130 seconds per iteration)
- Threads: 6

| Iter | Primal Obj | Dual Obj | Primal Res | Dual Res | Compl | Time |
|------|-------------|-----------|-------------|-----------|--------|-------|
| 0    | 8154.07     | -36998.39 | 9.31e+05    | 1.10e-03  | 2.22e+03 | 241s |
| 1    | 9087.92     | -481563.51| 6.18e+05    | 1.49e+00  | 1.37e+03 | 751s |

Barrier performed 1 iterations in 866.61s (670.60 work units)  
**Barrier solve interrupted** - model solved by another algorithm  
**Solved with dual simplex**

**Root relaxation**: objective **93.16932**, 31380 iterations, 790.26s (805.07 work units)


Explored 1 nodes (35349 simplex iterations) in 900.89 seconds (1050.97 work units)  
**Thread count**: 8 (of 8 available processors)  
**Solution count**: 1  
**Time limit reached**

**Best objective**: `906.000`  
**Best bound**: `93.16932291014`  
**Gap**: `89.7164%`  

---

### 📌 Makespan: **906.00**

## Completion Times (C<sub>ik</sub>)

| Job | Machine 0 | Machine 1 | Machine 2 | Machine 3 | Machine 4 |
|-----|-----------|-----------|-----------|-----------|-----------|
| 0   | 482.00    | 638.00    | 714.00    | 810.00    | 906.00    |
| 1   | 467.00    | 553.00    | 663.00    | 703.00    | 851.00    |
| 2   | 421.00    | 511.00    | 608.00    | 683.00    | 779.00    |
| 3   | 360.00    | 373.00    | 464.00    | 530.00    | 598.00    |
| 4   | 328.00    | 340.00    | 433.00    | 530.00    | 564.00    |
| 5   | 296.00    | 331.00    | 427.00    | 480.00    | 530.00    |
| 6   | 237.00    | 302.00    | 364.00    | 447.00    | 458.00    |
| 7   | 169.00    | 207.00    | 247.00    | 366.00    | 436.00    |
| 8   | 122.00    | 175.00    | 223.00    | 318.00    | 397.00    |
| 9   | 85.00     | 98.00     | 134.00    | 145.00    | 168.00    |

---

## Job Schedule (y[i,k,t]=1)

| Job | Machine 0     | Machine 1     | Machine 2     | Machine 3     | Machine 4     |
|-----|---------------|---------------|---------------|---------------|---------------|
| 0   | Periods 468-482 | Periods 554-638 | Periods 664-714 | Periods 715-810 | Periods 852-906 |
| 1   | Periods 422-467 | Periods 512-553 | Periods 609-663 | Periods 684-703 | Periods 780-851 |
| 2   | Periods 361-421 | Periods 422-511 | Periods 512-608 | Periods 609-683 | Periods 684-779 |
| 3   | Periods 329-360 | Periods 361-373 | Periods 434-464 | Periods 481-512 | Periods 565-598 |
| 4   | Periods 297-328 | Periods 332-340 | Periods 428-433 | Periods 530       | Periods 531-564 |
| 5   | Periods 238-296 | Periods 303-331 | Periods 365-427 | Periods 448-480 | Periods 481-530 |
| 6   | Periods 170-237 | Periods 238-302 | Periods 303-364 | Periods 367-447 | Periods 448-458 |
| 7   | Periods 123-169 | Periods 176-207 | Periods 224-247 | Periods 319-366 | Periods 398-436 |
| 8   | Periods 86-122  | Periods 123-175 | Periods 176-223 | Periods 224-318 | Periods 319-397 |
| 9   | Periods 1-85   | Periods 86-98  | Periods 99-134 | Periods 135-145 | Periods 146-168 |


---

### Precedence Order (z[i,j] = 1 → i before j)

| Job i (Before) | Job j (After) |
|----------------|---------------|
| 0              | 1             |
| 0              | 2             |
| 0              | 3             |
| 0              | 4             |
| 0              | 5             |
| 0              | 6             |
| 0              | 7             |
| 0              | 8             |
| 0              | 9             |
| 1              | 2             |
| 1              | 3             |
| 1              | 4             |
| 1              | 5             |
| 1              | 6             |
| 1              | 7             |
| 1              | 8             |
| 1              | 9             |
| 2              | 3             |
| 2              | 4             |
| 2              | 5             |
| 2              | 6             |
| 2              | 7             |
| 2              | 8             |
| 2              | 9             |
| 3              | 4             |
| 3              | 5             |
| 3              | 6             |
| 3              | 7             |
| 3              | 8             |
| 3              | 9             |
| 4              | 5             |
| 4              | 6             |
| 4              | 7             |
| 4              | 8             |
| 4              | 9             |
| 5              | 6             |
| 5              | 7             |
| 5              | 8             |
| 5              | 9             |
| 6              | 7             |
| 6              | 8             |
| 6              | 9             |
| 7              | 8             |
| 7              | 9             |
| 8              | 9             |




In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR20_5_1_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 1800  # 30 minutes = 1800 seconds
model.setParam('MIPGap',0.25) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR20_5_2_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 900  # 15 minutes = 900 seconds
model.setParam('MIPGap',0.20) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR10_5_3_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 900  # 15 minutes = 900 seconds
model.setParam('MIPGap',0.20) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR20_5_4_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 900  # 15 minutes = 900 seconds
model.setParam('MIPGap',0.20) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")


In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np


# --- Data ---
def read_instance(filename):
    """
    Reads job and machine data from file.
    Format: First line is n (jobs) and m (machines),
    followed by n lines with machine-time pairs.
    """
    with open(filename, 'r') as f:
        lines = f.readlines()
        n, m = map(int, lines[0].strip().split())
        processing_times = np.zeros((n, m), dtype=int)
        for i in range(n):
            data = list(map(int, lines[i + 1].strip().split()))
            for j in range(0, len(data), 2):
                machine = data[j]
                time = data[j + 1]
                processing_times[i][machine] = time
    return n, m, processing_times

# --- Load instance ---
filename = '/Users/nasiyapervez/Downloads/project directory/VFR20_5_5_Gap.txt'
try:
    n, m, p_ik = read_instance(filename)
    print(f"Loaded: {n} jobs, {m} machines")
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# --- Parameters ---
T = int(n * np.max(p_ik))  # Time horizon as upper bound (n * max processing time)
M = T                # Big-M for constraints

# --- Model ---
model = gp.Model("Bowman_Permutation_Flow_Shop")

# --- Decision Variables ---
y = model.addVars(n, m, T, vtype=GRB.BINARY, name="y")        # Binary: job i on machine k at time t
C_ik = model.addVars(n, m, vtype=GRB.CONTINUOUS, name="C_ik") # Completion time of job i on machine k
z_ij = model.addVars(n, n, vtype=GRB.BINARY, name="z")        # Job order: i before j
C_max = model.addVar(vtype=GRB.CONTINUOUS, name="C_max")      # Makespan

# --- Slack variables for processing time relaxation ---
# slack_p = model.addVars(n, m, lb=0, vtype=GRB.CONTINUOUS, name="slack_p")

# --- Binary variables and slack for precedence relaxation ---
# relax_prec = model.addVars(n, m-1, vtype=GRB.BINARY, name="relax_prec")

# --- Objective: Minimize Makespan + penalties for slacks ---
penalty_p = 1000      # penalty weight for processing time slack
penalty_prec = 10000  # penalty for relaxing precedence

model.setObjective(
    C_max,
    GRB.MINIMIZE
)

# --- Constraints ---
# 1. Machine capacity: only one job at a time on each machine
for k in range(m):
    for t in range(T):
        model.addConstr(
            gp.quicksum(y[i, k, t] for i in range(n)) <= 1,
            name=f"Capacity_m{k}_t{t}"
        )

# 2. Processing time: job i must run at least p_ik[i][k] time units on machine k
for i in range(n):
    for k in range(m):
        model.addConstr(
            gp.quicksum(y[i, k, t] for t in range(T)) >= p_ik[i, k],
            name=f"Processing_j{i}_m{k}"
        )

# 3. Precedence: job i on machine k+1 cannot start before it finishes on k
for i in range(n):
    for k in range(m - 1):
        for t in range(T):
            model.addConstr(
                p_ik[i, k] * y[i, k + 1, t] <= gp.quicksum(y[i, k, l] for l in range(t)),
                name=f"Precedence_j{i}_m{k}_t{t}"
            )

# 4. Contiguous processing blocks
for i in range(n):
    for k in range(m):
        for t in range(T - 1):
            sum_after = gp.quicksum(y[i, k, l] for l in range(t + 2, T))
            model.addConstr(
                p_ik[i, k] * (y[i, k, t] - y[i, k, t + 1]) + sum_after <= p_ik[i, k],
                name=f"Contiguous_j{i}_m{k}_t{t}"
            )

# 5. Completion time
for i in range(n):
    for k in range(m):
        for t in range(T):
            model.addConstr(
                y[i, k, t] * (t + 1) <= C_ik[i, k],
                name=f"Completion_j{i}_m{k}_t{t}"
            )

# 6. Sequencing constraint 1
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[i, k] >= C_ik[j, k] - M * z_ij[i, j],
                    name=f"Seq1_j{i}_j{j}_m{k}"
                )

# 7. Sequencing constraint 2
for i in range(n):
    for j in range(n):
        if i < j:
            for k in range(m):
                model.addConstr(
                    C_ik[j, k] >= C_ik[i, k] - M * (1 - z_ij[i, j]),
                    name=f"Seq2_j{i}_j{j}_m{k}"
                )

# 8. Makespan constraint
for i in range(n):
    model.addConstr(
        C_max >= C_ik[i, m - 1],
        name=f"Makespan_j{i}"
    )


# --- Optimize with time limit and MIP Gap ---
model.Params.TimeLimit = 900  # 15 minutes = 900 seconds
model.setParam('MIPGap',0.20) 
model.optimize()

# --- Results ---
if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT or model.status == GRB.SUBOPTIMAL:
    try:
        print(f"\n Makespan: {C_max.X:.2f}")
    except AttributeError:
        print("\n  Makespan: Not available")

    print("\n Completion Times (C_ik):")
    for i in range(n):
        for k in range(m):
            try:
                print(f"  Job {i}, Machine {k}: C_ik = {C_ik[i, k].X:.2f}")
            except AttributeError:
                print(f"  Job {i}, Machine {k}: Not available")

    print("\n Job Schedule (y[i,k,t]=1):")
    for i in range(n):
        for k in range(m):
            active = []
            for t in range(T):
                try:
                    if y[i, k, t].X > 0.5:
                        active.append(t + 1)
                except AttributeError:
                    pass
            if active:
                blocks = []
                start = end = active[0]
                for t in active[1:]:
                    if t == end + 1:
                        end = t
                    else:
                        blocks.append(f"{start}-{end}" if start != end else f"{start}")
                        start = end = t
                blocks.append(f"{start}-{end}" if start != end else f"{start}")
                print(f"  Job {i}, Machine {k}: Periods {', '.join(blocks)}")
            else:
                print(f"  Job {i}, Machine {k}: None")

    print("\n Precedence Order (z[i,j]=1 → i before j):")
    for i in range(n):
        for j in range(n):
            if i < j:
                try:
                    order = "→" if z_ij[i, j].X > 0.5 else "←"
                    print(f"  Job {i} {order} Job {j}")
                except AttributeError:
                    print(f"  Job {i} ? Job {j}")
else:
    print(f"\n Optimization Status: {model.status}")
    if model.status == GRB.INFEASIBLE:
        print("Model is infeasible.")
    elif model.status == GRB.UNBOUNDED:
        print("Model is unbounded.")
